# CosineAnnealingWarmRestartsDecay Visualization

This notebook visualizes our custom learning rate scheduler that combines:
- **Cosine Annealing**: Smooth LR transitions within each cycle
- **Warm Restarts**: Multiple training cycles for better exploration
- **LR Decay**: Reduces max LR after each restart for stability in later phases

This scheduler is designed to align with progressive augmentation, where:
- Early phases (high LR) → model learns basic features with no/light augmentation
- Later phases (lower LR) → model fine-tunes with stronger augmentation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set style for better visualization
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

In [ ]:
class CosineAnnealingWarmRestartsDecay:
    """
    Cosine annealing scheduler with warm restarts and optional LR decay.

    Similar to PyTorch's CosineAnnealingWarmRestarts but with decay factor
    that reduces the max LR after each restart. This is useful for progressive
    augmentation where early phases need high LR and later phases benefit from
    more stable, lower LR.

    Args:
        base_lr: Base learning rate
        T_0: Number of epochs for the first restart cycle
        T_mult: Factor to increase cycle length after each restart (default: 1)
        eta_min: Minimum learning rate (default: 1e-6)
        decay: Factor to multiply max LR after each restart (default: 1.0, no decay)
               e.g., decay=0.8 means max LR becomes 80% after each restart
    """

    def __init__(self, base_lr, T_0, T_mult=1, eta_min=1e-6, decay=1.0):
        self.base_lr = base_lr
        self.T_0 = T_0
        self.T_mult = T_mult
        self.eta_min = eta_min
        self.decay = decay

    def get_lr(self, epoch):
        """Calculate learning rate for given epoch"""
        # Determine which cycle we're in and position within cycle
        if epoch >= self.T_0:
            if self.T_mult == 1:
                cycle = epoch // self.T_0
                T_cur = epoch % self.T_0
                T_i = self.T_0
            else:
                # Handle T_mult > 1 (increasing cycle lengths)
                n = int(np.log((epoch / self.T_0 * (self.T_mult - 1) + 1)) / np.log(self.T_mult))
                cycle = n
                T_cur = epoch - self.T_0 * (self.T_mult ** n - 1) // (self.T_mult - 1)
                T_i = self.T_0 * self.T_mult ** n
        else:
            T_cur = epoch
            cycle = 0
            T_i = self.T_0

        # Calculate decay factor based on cycle number
        decay_factor = self.decay ** cycle

        # Cosine annealing within current cycle
        lr = self.eta_min + (self.base_lr * decay_factor - self.eta_min) * 0.5 * (1 + np.cos(np.pi * T_cur / T_i))
        return lr, cycle

## 1. Basic Visualization: decay=1.0 vs decay=0.8

Compare standard cosine restarts (no decay) with our decaying version.

In [ ]:
# Parameters
epochs = 100
base_lr = 1e-4
T_0 = 25  # epochs // 4
eta_min = 1e-6

# Create schedulers
scheduler_no_decay = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=1.0, eta_min=eta_min)
scheduler_decay = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=0.8, eta_min=eta_min)

# Collect LR values
lrs_no_decay = [scheduler_no_decay.get_lr(e)[0] for e in range(epochs)]
lrs_decay = [scheduler_decay.get_lr(e)[0] for e in range(epochs)]
cycles = [scheduler_decay.get_lr(e)[1] for e in range(epochs)]

# Plot
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(range(epochs), lrs_no_decay, 'b-', linewidth=2, label='decay=1.0 (no decay)', alpha=0.7)
ax.plot(range(epochs), lrs_decay, 'r-', linewidth=2, label='decay=0.8')

# Add vertical lines at restart points
for i in range(1, 4):
    ax.axvline(x=i * T_0, color='gray', linestyle='--', alpha=0.5)
    ax.text(i * T_0 + 0.5, base_lr * 1.05, f'Restart {i}', fontsize=10, color='gray')

# Add augmentation level annotations
aug_colors = ['#e8f5e9', '#c8e6c9', '#a5d6a7', '#81c784']
aug_labels = ['Level 0\n(No Aug)', 'Level 1\n(Light)', 'Level 2\n(Medium)', 'Level 3\n(Strong)']
for i in range(4):
    ax.axvspan(i * T_0, (i + 1) * T_0, alpha=0.3, color=aug_colors[i])
    ax.text(i * T_0 + T_0/2, base_lr * 0.5, aug_labels[i], ha='center', fontsize=9, color='darkgreen')

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('CosineAnnealingWarmRestartsDecay: Effect of Decay Factor', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.set_xlim(0, epochs)
ax.set_ylim(0, base_lr * 1.15)

plt.tight_layout()
plt.show()

## 2. Different Decay Values

Visualize how different decay values affect the learning rate schedule.

In [ ]:
# Compare different decay values
decay_values = [1.0, 0.9, 0.8, 0.7, 0.5]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(decay_values)))

fig, ax = plt.subplots(figsize=(14, 6))

for decay, color in zip(decay_values, colors):
    scheduler = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=decay, eta_min=eta_min)
    lrs = [scheduler.get_lr(e)[0] for e in range(epochs)]
    ax.plot(range(epochs), lrs, color=color, linewidth=2, label=f'decay={decay}')

# Add vertical lines at restart points
for i in range(1, 4):
    ax.axvline(x=i * T_0, color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Learning Rate', fontsize=12)
ax.set_title('Effect of Different Decay Values on LR Schedule', fontsize=14)
ax.legend(loc='upper right', fontsize=11)
ax.set_xlim(0, epochs)
ax.set_ylim(0, base_lr * 1.1)

plt.tight_layout()
plt.show()

## 3. T_mult Effect (Increasing Cycle Lengths)

In [ ]:
# Compare T_mult values
epochs_long = 200

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# T_mult = 1 (constant cycle length)
scheduler_t1 = CosineAnnealingWarmRestartsDecay(base_lr, T_0=25, T_mult=1, decay=0.8, eta_min=eta_min)
lrs_t1 = [scheduler_t1.get_lr(e)[0] for e in range(epochs_long)]

axes[0].plot(range(epochs_long), lrs_t1, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Learning Rate', fontsize=12)
axes[0].set_title('T_mult=1 (Constant Cycle Length = 25)', fontsize=13)
axes[0].set_xlim(0, epochs_long)
axes[0].set_ylim(0, base_lr * 1.1)

# T_mult = 2 (doubling cycle length)
scheduler_t2 = CosineAnnealingWarmRestartsDecay(base_lr, T_0=20, T_mult=2, decay=0.85, eta_min=eta_min)
lrs_t2 = [scheduler_t2.get_lr(e)[0] for e in range(epochs_long)]

axes[1].plot(range(epochs_long), lrs_t2, 'r-', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('T_mult=2 (Doubling Cycle: 20→40→80→...)', fontsize=13)
axes[1].set_xlim(0, epochs_long)
axes[1].set_ylim(0, base_lr * 1.1)

plt.tight_layout()
plt.show()

## 4. Max LR Values Per Cycle

Show how the peak learning rate decreases with each restart.

In [ ]:
# Calculate max LR for each cycle with decay=0.8
decay = 0.8
num_cycles = 8

max_lrs = [base_lr * (decay ** i) for i in range(num_cycles)]

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(range(num_cycles), max_lrs, color=plt.cm.Reds(np.linspace(0.9, 0.3, num_cycles)), edgecolor='darkred')

# Add value labels on bars
for i, (bar, lr) in enumerate(zip(bars, max_lrs)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + base_lr*0.02, 
            f'{lr:.2e}', ha='center', va='bottom', fontsize=10)

ax.set_xlabel('Cycle Number', fontsize=12)
ax.set_ylabel('Max Learning Rate', fontsize=12)
ax.set_title(f'Peak LR Per Cycle (base_lr={base_lr:.0e}, decay={decay})', fontsize=14)
ax.set_xticks(range(num_cycles))
ax.set_xticklabels([f'Cycle {i}' for i in range(num_cycles)])

plt.tight_layout()
plt.show()

print("\nMax LR reduction per cycle:")
for i, lr in enumerate(max_lrs):
    pct = (lr / base_lr) * 100
    print(f"  Cycle {i}: {lr:.2e} ({pct:.1f}% of base LR)")

## 5. Log Scale Visualization

Useful for seeing the full range from max LR to eta_min.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

scheduler = CosineAnnealingWarmRestartsDecay(base_lr, T_0, decay=0.8, eta_min=eta_min)
lrs = [scheduler.get_lr(e)[0] for e in range(epochs)]

# Linear scale
axes[0].plot(range(epochs), lrs, 'b-', linewidth=2)
axes[0].axhline(y=eta_min, color='r', linestyle='--', alpha=0.5, label=f'eta_min={eta_min:.0e}')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Learning Rate', fontsize=12)
axes[0].set_title('Linear Scale', fontsize=13)
axes[0].legend()
axes[0].set_xlim(0, epochs)

# Log scale
axes[1].plot(range(epochs), lrs, 'b-', linewidth=2)
axes[1].axhline(y=eta_min, color='r', linestyle='--', alpha=0.5, label=f'eta_min={eta_min:.0e}')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Learning Rate (log scale)', fontsize=12)
axes[1].set_title('Log Scale', fontsize=13)
axes[1].set_yscale('log')
axes[1].legend()
axes[1].set_xlim(0, epochs)

plt.tight_layout()
plt.show()

## Summary

**CosineAnnealingWarmRestartsDecay** combines three key features:

1. **Cosine Annealing**: Smooth, gradual LR reduction within each cycle (better than step decay)

2. **Warm Restarts**: Multiple exploration phases allow escaping local minima

3. **Decay Factor**: Reduces max LR after each restart, providing:
   - High LR in early phases → fast learning of basic features
   - Lower LR in later phases → stable fine-tuning with strong augmentation

This aligns perfectly with **progressive augmentation** where:
- Cycle 0 (highest LR): No augmentation → learn basic patterns
- Cycle 1 (80% LR): Light augmentation → improve robustness  
- Cycle 2 (64% LR): Medium augmentation → further regularization
- Cycle 3 (51% LR): Strong augmentation → final fine-tuning